# 008 · Dataclasses

> **Syntax › Classes & OOP** · Python 3.14

The `@dataclass` decorator turns a class with annotated fields into a full-featured data type: it generates `__init__`, `__repr__`, `__eq__` and, on request, ordering, immutability, hashing and `__slots__`. It is Python's closest equivalent to C# `record` types.

**In this notebook you will learn:**

- how `@dataclass` compares to C# `record` / `record struct`
- defaults, and `field(default_factory=...)` to avoid the mutable-default trap
- `frozen=True` (immutable and hashable), `order=True`, `slots=True`, `kw_only=True`
- `dataclasses.replace` as the counterpart of the `with` expression
- validation in `__post_init__`, and converting with `asdict`
- a brief comparison with `typing.NamedTuple`

## C# mental model

| C# | Python |
|---|---|
| `record Point(int X, int Y);` | `@dataclass(frozen=True) class Point: x: int; y: int` |
| `class` with properties + ctor | `@dataclass class ...` (mutable by default) |
| `record struct` | (no value types) — `@dataclass(frozen=True, slots=True)` is the lightweight equivalent |
| `p with { X = 5 }` | `dataclasses.replace(p, x=5)` |
| `{ get; init; }` | `frozen=True` (all fields) |
| compiler-generated `ToString`/`Equals`/`GetHashCode` | generated `__repr__`/`__eq__`/`__hash__` |
| `IComparable` by hand | `order=True` |
| validation in the constructor | `__post_init__` |

The key mental shift: a dataclass is a **regular class** with generated methods — you can add methods, properties and inheritance as usual. The field list is read from the class's **type annotations**.

## A first dataclass

```csharp
// C#
public record Product(string Name, decimal Price, int Quantity = 0);
```

In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float
    quantity: int = 0

    def total(self):                   # ordinary methods work as usual
        return self.price * self.quantity

p = Product("pen", 1.5, 4)
print(p)                               # generated __repr__
print(p == Product("pen", 1.5, 4))     # generated __eq__ — value equality
print(p.total())

Compare with what you'd write by hand:

```python
class Product:
    def __init__(self, name: str, price: float, quantity: int = 0):
        self.name = name
        self.price = price
        self.quantity = quantity

    def __repr__(self):
        return f"Product(name={self.name!r}, price={self.price!r}, quantity={self.quantity!r})"

    def __eq__(self, other):
        if other.__class__ is not self.__class__:
            return NotImplemented
        return (self.name, self.price, self.quantity) == (other.name, other.price, other.quantity)
```

⚠️ **Gotcha:** unlike a C# `record class`, a plain `@dataclass` is **mutable** — fields can be reassigned. And like all Python type hints, the annotations are **not enforced** at runtime:

In [ ]:
p.price = 2.0             # allowed — mutable by default
odd = Product("x", "not a number")
print(p, odd)             # no runtime type check

⚠️ **Gotcha:** a field **must** have an annotation to become a field. A bare `x = 0` in the class body is just a class attribute and is ignored by the dataclass machinery. Also, fields without defaults must come before fields with defaults (like C# optional parameters) — unless you use `kw_only` (below).

## Defaults and the mutable default trap

Mutable defaults like `tags: list = []` would be **shared** by every instance — the same trap as mutable default arguments and mutable class attributes. Dataclasses refuse them outright, and you use `field(default_factory=...)` instead, which calls the factory for each new instance:

```csharp
// C#
public record Order(string Id) { public List<string> Items { get; init; } = new(); }
```

In [ ]:
from dataclasses import field

try:
    @dataclass
    class BadOrder:
        items: list = []
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

@dataclass
class Order:
    id: str
    items: list[str] = field(default_factory=list)   # fresh list per instance

o1, o2 = Order("A1"), Order("A2")
o1.items.append("pen")
print(o1, o2)

`field()` also lets you customise a single field, e.g. exclude it from `repr` or equality:

In [ ]:
@dataclass
class User:
    name: str
    password_hash: str = field(repr=False)             # hide from repr
    login_count: int = field(default=0, compare=False) # ignore in ==

a = User("alice", "x1f9")
b = User("alice", "x1f9", login_count=5)
print(a)
print(a == b)    # True — login_count not compared

## `frozen=True` — immutable and hashable

A frozen dataclass raises `FrozenInstanceError` on assignment, like a C# record with `init`-only properties. Because its fields can't change, it is also given a `__hash__`, so it can be used as a dict key or set member.

In [ ]:
from dataclasses import FrozenInstanceError

@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(1, 2)
try:
    p.x = 10
except FrozenInstanceError as e:
    print(f"{type(e).__name__}: {e}")

print({Point(1, 2), Point(1, 2)})        # hashable — duplicates removed
print({Point(0, 0): "origin"})

A regular (non-frozen) dataclass with the default `eq=True` sets `__hash__` to `None` — just like a hand-written class that defines `__eq__` *(see Classes & OOP › 007 · Dunder Methods)*:

In [ ]:
try:
    hash(Product("pen", 1.5))
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** `frozen` is shallow. A frozen dataclass holding a `list` can still have that list mutated — and then it is no longer hashable. Use tuples / `frozenset` for collection fields of frozen classes.

## `dataclasses.replace` — the `with` expression

Immutable objects are "changed" by creating a modified copy:

```csharp
// C#
var moved = p with { X = 5 };
```

In [ ]:
from dataclasses import replace

moved = replace(p, x=5)
print(p, moved)

Python 3.13 added `copy.replace()`, a general version that also works for named tuples, `datetime` objects and any class defining `__replace__` — dataclasses support it automatically:

In [ ]:
import copy
print(copy.replace(p, y=99))

## `order=True` — comparison operators

With `order=True`, the dataclass generates `<`, `<=`, `>`, `>=`, comparing fields **as a tuple in declaration order** — no need to implement `IComparable<T>`:

In [ ]:
@dataclass(order=True, frozen=True)
class Version:
    major: int
    minor: int
    patch: int = 0

versions = [Version(1, 10), Version(1, 2, 5), Version(0, 9)]
print(sorted(versions))
print(Version(2, 0) > Version(1, 99))

To order by something other than declaration order, put a sort key field first, or use `sorted(..., key=...)` instead.

## `slots=True` and `kw_only=True` (Python 3.10+)

- `slots=True` generates `__slots__` — smaller, slightly faster instances that reject unknown attributes *(see Classes & OOP › 001 · Classes and Objects)*.
- `kw_only=True` makes every field keyword-only in `__init__`, like requiring named arguments. It also lifts the "defaults must come last" restriction.

In [ ]:
@dataclass(slots=True, kw_only=True)
class Config:
    host: str = "localhost"
    port: int
    debug: bool = False

cfg = Config(port=8080, debug=True)
print(cfg)

try:
    Config("example", 80)               # positional args not allowed
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

try:
    cfg.timeout = 30                    # slots: no new attributes
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

You can also make only some fields keyword-only with `field(kw_only=True)`, or with a `_: KW_ONLY` separator field.

## Validation and derived fields — `__post_init__`

The generated `__init__` calls `__post_init__` (if defined) after assigning fields. Use it for validation or computing derived fields — the role a C# record's explicit constructor or property initialiser plays.

```csharp
// C#
public record Interval(int Start, int End)
{
    public int Start { get; } = Start <= End ? Start : throw new ArgumentException("start > end");
}
```

In [ ]:
@dataclass(frozen=True)
class Interval:
    start: int
    end: int
    length: int = field(init=False)     # computed, not a constructor parameter

    def __post_init__(self):
        if self.start > self.end:
            raise ValueError(f"start ({self.start}) > end ({self.end})")
        # frozen: normal assignment is blocked, so use object.__setattr__
        object.__setattr__(self, "length", self.end - self.start)

print(Interval(2, 7))
try:
    Interval(5, 1)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

Validation only runs in `__init__`. On a mutable dataclass, later assignments bypass it — use a `frozen` class, or a property *(see Classes & OOP › 003 · Properties)*.

## Converting: `asdict`, `astuple`, `fields`

`asdict` recursively converts a dataclass (including nested dataclasses, lists and dicts) to plain dicts — handy for JSON serialisation:

In [ ]:
import json
from dataclasses import asdict, astuple, fields

@dataclass
class Line:
    start: Point
    end: Point

line = Line(Point(0, 0), Point(3, 4))
print(asdict(line))
print(json.dumps(asdict(line)))
print(astuple(line))
print([f.name for f in fields(line)])   # introspection — like reflection over properties

## Inheritance

Dataclasses can inherit from each other; fields are collected from base classes in order. A subclass may add methods, override defaults or add new fields.

In [ ]:
@dataclass
class Animal:
    name: str

@dataclass
class Dog(Animal):
    breed: str = "unknown"

print(Dog("rex", "beagle"), isinstance(Dog("rex"), Animal))

## Brief comparison: `typing.NamedTuple`

`NamedTuple` creates an immutable **tuple subclass** with named fields. It is lighter and supports unpacking and indexing, but it *is* a tuple — so it compares equal to plain tuples with the same values, which is rarely what you want for domain objects.

In [ ]:
from typing import NamedTuple

class Coord(NamedTuple):
    lat: float
    lon: float

c = Coord(51.5, -0.1)
lat, lon = c                        # tuple unpacking
print(c, c[0], c.lon)
print(c == (51.5, -0.1))            # True — tuple semantics!
print(c._replace(lat=0.0))

| | `@dataclass` | `NamedTuple` |
|---|---|---|
| Mutable | yes (default) / `frozen=True` | never |
| Iterable / indexable | no | yes |
| Equal to plain tuples | no | yes |
| Methods, inheritance, `__post_init__` | full support | limited |
| Best for | domain objects, records | lightweight return values, tuple replacements |

*(see Collections › 002 · Tuples)*

## Summary

| Task | C# | Python |
|---|---|---|
| Data type | `record Product(string Name, ...)` | `@dataclass class Product: name: str` |
| Immutable | `record` / `init` | `@dataclass(frozen=True)` |
| Value equality | generated `Equals` | generated `__eq__` |
| Hashable | generated `GetHashCode` | `frozen=True` (or `unsafe_hash=True`) |
| Non-destructive mutation | `p with { X = 1 }` | `replace(p, x=1)` / `copy.replace` |
| Collection default | `= new()` | `field(default_factory=list)` |
| Ordering | implement `IComparable<T>` | `order=True` |
| Named arguments only | (no equivalent) | `kw_only=True` |
| Compact instances | `record struct` | `slots=True` |
| Validation | constructor body | `__post_init__` |
| Convert to dict | serializer / reflection | `asdict()` |

## Exercises

1. Create a frozen dataclass `Money` with `amount: int` and `currency: str = "EUR"`, validating in `__post_init__` that `amount >= 0`. Use it as a dict key.
2. Create a dataclass `ShoppingCart` with an `owner: str` and an `items: list[str]` field defaulting to an empty list. Prove two carts don't share their list.
3. Define `@dataclass(order=True) class Task` with `priority: int` and `title: str`, and sort a list of tasks. Then make `title` excluded from comparisons.
4. Given a frozen `Point(x, y)`, write a function `translate(p, dx, dy)` that returns a moved copy using `replace`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from dataclasses import dataclass, field, replace

# 1
@dataclass(frozen=True)
class Money:
    amount: int
    currency: str = "EUR"

    def __post_init__(self):
        if self.amount < 0:
            raise ValueError("amount must be >= 0")

prices = {Money(5): "cheap", Money(500): "pricey"}
print(prices[Money(5)])

# 2
@dataclass
class ShoppingCart:
    owner: str
    items: list[str] = field(default_factory=list)

c1, c2 = ShoppingCart("alice"), ShoppingCart("bob")
c1.items.append("apple")
print(c1.items, c2.items)     # ['apple'] []

# 3
@dataclass(order=True)
class Task:
    priority: int
    title: str = field(compare=False)

print(sorted([Task(3, "c"), Task(1, "a"), Task(2, "b")]))

# 4
@dataclass(frozen=True)
class Point:
    x: float
    y: float

def translate(p, dx, dy):
    return replace(p, x=p.x + dx, y=p.y + dy)

print(translate(Point(1, 1), 2, 3))   # Point(x=3, y=4)
```

</details>